# Retail sales: exploration and cleaning

Run cells from top to bottom. Keep missing CustomerIDs in sales totals; exclude them only from customer analysis.

In [1]:
from pathlib import Path

import pandas as pd

# Support opening Jupyter from either the project or notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / "data/raw/Online Retail.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_FILE.is_file():
    raise FileNotFoundError("Open this notebook from the project or notebooks folder.")

In [2]:
df = pd.read_excel(RAW_FILE)

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Inspect data quality

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [5]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(5268)

In [7]:
df[df.duplicated()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
587,536412,22273,FELTCRAFT DOLL MOLLY,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
589,536412,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,1,2010-12-01 11:49:00,3.75,17920.0,United Kingdom
594,536412,22141,CHRISTMAS CRAFT TREE TOP ANGEL,1,2010-12-01 11:49:00,2.10,17920.0,United Kingdom
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
600,536412,22569,FELTCRAFT CUSHION BUTTERFLY,2,2010-12-01 11:49:00,3.75,17920.0,United Kingdom


In [8]:
print("Quantity zero:", df.query("Quantity == 0").shape[0])
print("UnitPrice zero:", df.query("UnitPrice == 0").shape[0])
print("Quantity negative:", df.query("Quantity < 0").shape[0])
print("UnitPrice negative:", df.query("UnitPrice < 0").shape[0])

Quantity zero: 0
UnitPrice zero: 2515
Quantity negative: 10624
UnitPrice negative: 2


In [9]:
df.query("Quantity < 0").head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [10]:
df.query("UnitPrice < 0").head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [11]:
df.query("UnitPrice == 0").head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [12]:
is_cancelled = df["InvoiceNo"].astype(str).str.startswith("C")
print("Cancellation rows before cleaning:", df[is_cancelled].shape[0])
print(
    "Negative quantity without C invoice:",
    df[~is_cancelled].query("Quantity < 0").shape[0],
)

Cancellation rows before cleaning: 9288
Negative quantity without C invoice: 1336


## Remove exact duplicates and select positive sales

Credits and accounting adjustments remain in cleaned_transactions.csv. Sales totals are before credits, not net revenue or profit.

In [13]:
df_clean = df.drop_duplicates(keep="first").copy()

print("Remaining rows:", df_clean.shape[0])
print("Duplicate rows:", df_clean.duplicated().sum())

Remaining rows: 536641
Duplicate rows: 0


In [14]:
is_cancelled = df_clean["InvoiceNo"].astype(str).str.startswith("C")
sales_df = df_clean[~is_cancelled].query("Quantity > 0 and UnitPrice > 0").copy()

print("Total sales rows:", sales_df.shape[0])

Total sales rows: 524878


In [15]:
sales_df["SalesAmount"] = sales_df["Quantity"] * sales_df["UnitPrice"]

In [16]:
sales_df[["Quantity", "UnitPrice", "SalesAmount"]].head(5)

,Quantity,UnitPrice,SalesAmount
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


## Sales, countries, products and months

In [17]:
total_sales = sales_df["SalesAmount"].sum()
total_quantity = sales_df["Quantity"].sum()
total_invoices = sales_df["InvoiceNo"].nunique()

print("Total sales amount:", round(total_sales, 2))
print("Total quantity sold:", total_quantity)
print("Total unique invoices:", total_invoices)

Total sales amount: 10642110.8
Total quantity sold: 5572420
Total unique invoices: 19960


In [18]:
country_sales = sales_df.groupby("Country")["SalesAmount"].sum()
country_sales = country_sales.sort_values(ascending=False)
country_sales.head(10)

Country
United Kingdom    9001744.094
Netherlands        285446.340
EIRE               283140.520
Germany            228678.400
France             209625.370
Australia          138453.810
Spain               61558.560
Switzerland         57067.600
Belgium             41196.340
Sweden              38367.830
Name: SalesAmount, dtype: float64

In [19]:
# Exclude postage and manual entries only from the product ranking.
products_df = sales_df[~sales_df["StockCode"].isin(["DOT", "POST", "M"])]
product_sales = products_df.groupby("StockCode")["SalesAmount"].sum()

In [20]:
product_sales = product_sales.sort_values(ascending=False).head(10)

In [21]:
product_sales

StockCode
22423     174156.54
23843     168469.60
85123A    104462.75
47566      99445.23
85099B     94159.81
23166      81700.92
23084      66870.03
22086      64875.59
84879      58927.62
79321      54096.36
Name: SalesAmount, dtype: float64

In [22]:
sales_df["Year"] = sales_df["InvoiceDate"].dt.year
sales_df["Month"] = sales_df["InvoiceDate"].dt.month

monthly_sales = sales_df.groupby(["Year", "Month"])["SalesAmount"].sum()

monthly_sales = monthly_sales.sort_index()

monthly_sales

Year  Month
2010  12        821452.730
2011  1         689811.610
      2         522545.560
      3         716215.260
      4         536968.491
      5         769296.610
      6         760547.010
      7         718076.121
      8         757841.380
      9        1056435.192
      10       1151263.730
      11       1503866.780
      12        637790.330
Name: SalesAmount, dtype: float64

In [23]:
sales_df["InvoiceDate"].min()

Timestamp('2010-12-01 08:26:00')

In [24]:
sales_df["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

In [25]:
sales_df.to_csv(PROCESSED_DIR / "cleaned_sales.csv", index=False)

## Customer purchasing

A repeat customer has more than one distinct invoice during the dataset period.

In [26]:
customer_sales = sales_df.dropna(subset=["CustomerID"]).copy()
total_customers = customer_sales["CustomerID"].nunique()
print("Known customers:", total_customers)

Known customers: 4338


In [27]:
customer_invoices = (
    customer_sales.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index(name="InvoiceCount")
)

one_time_customers = customer_invoices.query("InvoiceCount == 1").shape[0]
repeat_customers = customer_invoices.query("InvoiceCount > 1").shape[0]

print("One-time customers:", one_time_customers)
print("Repeat customers:", repeat_customers)

One-time customers: 1493
Repeat customers: 2845


In [28]:
repeat_percentage = repeat_customers / total_customers * 100 if total_customers else 0
print(f"Repeat customers percentage: {repeat_percentage:.2f}%")

Repeat customers percentage: 65.58%


In [29]:
top_customers = (
    customer_sales.groupby("CustomerID")["SalesAmount"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_customers

CustomerID
14646.0    280206.02
18102.0    259657.30
17450.0    194390.79
16446.0    168472.50
14911.0    143711.17
12415.0    124914.53
14156.0    117210.08
17511.0     91062.38
16029.0     80850.84
12346.0     77183.60
Name: SalesAmount, dtype: float64

## Cancellation credits

These are credit records, not returns matched to original purchases.

In [30]:
cancelled_rows = df_clean[df_clean["InvoiceNo"].astype(str).str.startswith("C")]

print("Cancellation rows:", cancelled_rows.shape[0])
print("Unique cancellation invoices:", cancelled_rows["InvoiceNo"].nunique())

Cancellation rows: 9251
Unique cancellation invoices: 3836


In [31]:
quantity_check = cancelled_rows.query("Quantity >= 0")
price_check = cancelled_rows.query("UnitPrice <= 0")

print("Quantity Check: ", quantity_check.shape[0])
print("Price Check: ", price_check.shape[0])

Quantity Check:  0
Price Check:  0


In [32]:
# Validate the sign before calculating the positive credit amount.
assert quantity_check.empty and price_check.empty, "Review cancellation signs first."
cancellation_amount = cancelled_rows["Quantity"] * cancelled_rows["UnitPrice"]
total_cancellation_amount = -cancellation_amount.sum()
print(f"Total cancellation amount (GBP): {total_cancellation_amount:,.2f}")

Total cancellation amount (GBP): 893,979.73


In [33]:
df_clean.to_csv(PROCESSED_DIR / "cleaned_transactions.csv", index=False)